# Week 4: LLM Benchmarking vs Week 3 ML

Cells in this notebook were taken from the tutorial written by Victoria Reynolds.

Steps:
- Load Week 3 test data
- Query LLM endpoint
- Parse outputs into structured predictions
- Compare ML vs each LLM
- Save raw and clean result CSV files

## Step 1: Imports

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

openai = __import__('openai')
OpenAI = openai.OpenAI

## Step 2: Paths and settings

In [4]:
ROOT = Path('..').resolve()

WEEK3_TEST_CSV = 'week3_artifacts/FTES_1hour_test.csv'
WEEK3_ML_RESULTS_CSV = 'week3_artifacts/xgb_test_predictions.csv'
WEEK3_ML_CONFIG_JSON = 'week3_artifacts/xgb_training_config.json'

with open(WEEK3_ML_CONFIG_JSON, "r", encoding="utf-8") as f:
    train_cfg = json.load(f)

TIME_COL = 'Time'
FEAT_COLS = train_cfg.get('feature_cols')
TRUE_VAL_COL = 'Injection Pressure'
ML_PRED_COL = 'Injection Pressure__pred'

MODEL_ENDPOINTS = [  # Only using gemma-3-12b-it
    # {'label': 'phi-3.5-mini-instruct', 'model_name': 'Phi-3.5-mini-instruct', 'host': 'localhost', 'port': 8000},
    # {'label': 'phi-mini-moe-instruct', 'model_name': 'Phi-mini-MoE-instruct', 'host': 'localhost', 'port': 8001},
    {'label': 'gemma-3-12b-it', 'model_name': 'gemma-3-12b-it', 'host': 'localhost', 'port': 8002},
]
API_KEY = 'EMPTY'
ITERATION_NUMBER = 1

PROMPT_DIR = ROOT / 'Prompts'
PROMPT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = ROOT / 'Results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_RESULTS_PATH = OUTPUT_DIR / f'llm_results_raw_{ITERATION_NUMBER}.csv'
CLEAN_RESULTS_PATH = OUTPUT_DIR / f'llm_results_clean_{ITERATION_NUMBER}.csv'

## Step 3: Load Week 3 files

In [3]:
test_df = pd.read_csv(WEEK3_TEST_CSV)
ml_df = pd.read_csv(WEEK3_ML_RESULTS_CSV)

needed_test = {TIME_COL, *FEAT_COLS, TRUE_VAL_COL}
needed_ml = {TIME_COL, ML_PRED_COL}
if needed_test - set(test_df.columns):
    raise ValueError(f'Missing in {WEEK3_TEST_CSV}: {sorted(needed_test - set(test_df.columns))}')
if needed_ml - set(ml_df.columns):
    raise ValueError(f'Missing in {WEEK3_ML_RESULTS_CSV}: {sorted(needed_ml - set(ml_df.columns))}')

df = test_df[[TIME_COL, *FEAT_COLS, TRUE_VAL_COL]].merge(
    ml_df[[TIME_COL, ML_PRED_COL]], on=TIME_COL, how='left'
)

print(f'Loaded rows in test set: {len(df)}')
# display(df.head())

Loaded rows in test set: 256


## Step 4: Prompt + parser helpers

What this does: defines a strict prompt format and a parser that turns model text into structured fields (prediction, confidence, parse status).

Why it matters: LLM responses are free-form by default, but evaluation requires deterministic, machine-readable outputs.

Principle: constrain generation and validate parsing. Reliability comes from clear output contracts plus defensive checks.

In [24]:
SYSTEM_PROMPT = """You are a careful Fracture Thermal Energy Storage time-series forecasting model. \n
Your task is to predict the Target metric at t+1 using only features measured at time t. \n
Do not use future information, external knowledge, or assumptions not present in the input. \n
Treat this as numeric regression, not classification. \n
Return exactly one JSON object and nothing else. \n\n

Rules: \n
Prediction must be in the correct units for the target. \n
Confidence must be in [0, 1]. \n
No markdown, no prose, no code fences, no extra keys. \n
Required JSON schema: \n
{
"prediction": <float>, 
"confidence": <0_to_1_float>
}
"""

few_shot_samples = [
    {
        't': "2/11/2025  11:00:00 PM",
        'input_features': {
            "Injection Pressure": 4981.6798,
            "Pressure_Differential": 4353.3743,
            "TC Interval Pressure": 4991.9235,
            "TN Interval Flow": 0.0008,
            "TL Collar Flow": 0.1707,
            "TC-TEC-INT-L": 43.2451,
            "TC-TEC-INT-L__lag_12h": 43.3222,
            "TC-TEC-INT-L__lag_24h": 43.1582,
            "TC-TEC-INT-L__roll_mean_12h": 43.3166,
            "TC-TEC-INT-L__roll_mean_24h": 43.2913, 
            "TN-TEC-BOT-U": 30.4198,
            "TN-TEC-BOT-U__lag_12h": 30.4127,
            "TN-TEC-BOT-U__lag_24h": 30.3488,
            "TN-TEC-BOT-U__roll_mean_12h": 30.4158,
            "TN-TEC-BOT-U__roll_mean_24h": 30.3955
        },
        'output_json': {'prediction': 4982.9503, 'confidence': 0.82}
    },
    {
        't': "2/12/2025  12:00:00 AM",
        'input_features': {
            "Injection Pressure": 4982.9503,
            "Pressure_Differential": 4354.5854,
            "TC Interval Pressure": 4992.9041,
            "TN Interval Flow": 0.0009,
            "TL Collar Flow": 0.1708,
            "TC-TEC-INT-L": 43.2164,
            "TC-TEC-INT-L__lag_12h": 43.3474,
            "TC-TEC-INT-L__lag_24h": 43.1934,
            "TC-TEC-INT-L__roll_mean_12h": 43.3102,
            "TC-TEC-INT-L__roll_mean_24h": 43.2949, 
            "TN-TEC-BOT-U": 30.4324,
            "TN-TEC-BOT-U__lag_12h": 30.4153,
            "TN-TEC-BOT-U__lag_24h": 30.1394,
            "TN-TEC-BOT-U__roll_mean_12h": 30.4164,
            "TN-TEC-BOT-U__roll_mean_24h": 30.3985
        },
        'output_json': {'prediction': 4982.1107, 'confidence': 0.91}
    }
]

def build_prompt(time_t, feature_names, features_t, version='v1', few_shot=None):
    features_json = json.dumps(feature_names)
    features_t = json.dumps(features_t.to_dict())

    if version == 'v1':
        prompt_lines = [
            'Forecast target: Injection Pressure at next 1 hour interval (t+1h)',
            f'Current timestamp: {time_t}',
            'Horizon: 1 hour', 
            'Target: Injection Pressure',
            'Units: pressure in psi',
            f'Use only the following feature snapshot at time t: {features_json}',
            'Important: Do not use future information to make predictions. Return JSON only. No explanation text.'
        ]
            
        if few_shot:
            for i, ex in enumerate(few_shot, start=1):
                prompt_lines.append(f'Example input at time t, {ex[t]}: {ex["input_features"]}')
                prompt_lines.append(f'Example output for prediction at time t+1h: {json.dumps(ex["output_json"])}')
        prompt_lines.extend([
            'Required output schema: {\"prediction\": \"<float>\", \"confidence\": <0_to_1_float>}',
            '',
            'Input:',
            str(features_t)
        ])
        return '\n'.join(prompt_lines)

    raise ValueError(f'Unknown version: {version}')
    

def parse_response(raw_text):
    output = {'llm_prediction': np.nan, 'llm_confidence': np.nan, 'parse_ok': False, 'parse_error': None}

    if not isinstance(raw_text, str) or not raw_text.strip():
        output['parse_error'] = 'empty_response'
        return output

    m = re.search(r'\{.*\}', raw_text.strip(), flags=re.DOTALL)
    candidate = m.group(0) if m else raw_text.strip()

    try:
        payload = json.loads(candidate)
    except Exception:
        output['parse_error'] = 'invalid_json'
        return output

    pred = payload.get('prediction', np.nan)
    try:
        pred = float(pred)
    except Exception:
        pred = np.nan

    conf = payload.get('confidence', np.nan)
    try:
        conf = float(conf)
    except Exception:
        conf = np.nan

    if not np.isnan(conf) and not (0 <= conf <= 1):
        conf = np.nan

    output['llm_prediction'] = pred
    output['llm_confidence'] = conf
    output['parse_ok'] = True
    return output

## Step 5: Single-row smoke test

What this does: tests one example on each model endpoint before running the full benchmark loop.

Why it matters: this catches endpoint/port issues early and confirms all models are reachable.

Principle: validate connectivity and output format for each model before large-scale evaluation.

In [25]:
def query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0):
    client = OpenAI(
        api_key=API_KEY,
        base_url=f"http://{endpoint_cfg['host']}:{endpoint_cfg['port']}/v1"
    )
    response = client.chat.completions.create(
        model=endpoint_cfg['model_name'],
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        temperature=temperature,
        seed=seed
    )
    return response.choices[0].message.content

example = df.iloc[0]
for endpoint_cfg in MODEL_ENDPOINTS:
    raw = query_llm(build_prompt(example['Time'], FEAT_COLS, example[FEAT_COLS]), endpoint_cfg, temperature=0.0, seed=0)
    parsed = parse_response(raw)
    print(f"\nModel: {endpoint_cfg['model_name']} (port {endpoint_cfg['port']})")
    print('Raw response:')
    print(raw)
    print('Parsed:')
    print(parsed)


Model: gemma-3-12b-it (port 8002)
Raw response:
```json
{
"prediction": 4981.8012,
"confidence": 0.95
}
```
Parsed:
{'llm_prediction': 4981.8012, 'llm_confidence': 0.95, 'parse_ok': True, 'parse_error': None}


In [ ]:
prompt_template_path = PROMPT_DIR / 'prompt_template.txt'
few_shot_path = PROMPT_DIR / 'few_shot_samples.json'
prompt_template_path.write_text(final_prompt_example, encoding='utf-8')
few_shot_path.write_text(json.dumps(few_shot_samples, indent=2), encoding='utf-8')

## Step 6: Full benchmark run

What this does: loops over all configured model endpoints and all test rows, storing raw responses and parsed predictions.

Why it matters: this produces side-by-side model outputs from a single consistent pipeline.

Principle: hold the dataset and prompt constant while varying models for a fair model-to-model comparison.

In [10]:
rows = []
for endpoint_cfg in MODEL_ENDPOINTS:
    print(f"\nRunning benchmark for {endpoint_cfg['model_name']} on port {endpoint_cfg['port']}...")
    for i, row in df.iterrows():
        raw = query_llm(
            make_prompt(str(row[TEXT_COL]), label_values),
            endpoint_cfg,
            temperature=0.0,
            seed=0
        )
        parsed = parse_response(raw, label_set)

        rows.append({
            ID_COL: row[ID_COL],
            'model_label': endpoint_cfg['label'],
            'model_name': endpoint_cfg['model_name'],
            'endpoint_port': endpoint_cfg['port'],
            'prompt_version': 'v1_strict_json',
            'ground_truth': str(row[LABEL_COL]),
            'ml_prediction': str(row[ML_PRED_COL]),
            'raw_response': raw,
            **parsed
        })

        if (i + 1) % 10 == 0:
            print(f"  Completed {i + 1}/{len(df)}")

results_df = pd.DataFrame(rows)
results_df.to_csv(RAW_RESULTS_PATH, index=False)
print(f'Saved raw CSV: {RAW_RESULTS_PATH}')
display(results_df.head())


Running benchmark for Phi-3.5-mini-instruct on port 8000...
  Completed 10/40
  Completed 20/40
  Completed 30/40
  Completed 40/40

Running benchmark for Phi-mini-MoE-instruct on port 8001...
  Completed 10/40
  Completed 20/40
  Completed 30/40
  Completed 40/40

Running benchmark for gemma-3-12b-it on port 8002...
  Completed 10/40
  Completed 20/40
  Completed 30/40
  Completed 40/40
Saved raw CSV: /home/jupyter-abby/Results/llm_results_raw_1.csv


,sample_id,model_label,model_name,endpoint_port,prompt_version,ground_truth,ml_prediction,raw_response,llm_prediction,llm_confidence,parse_ok,parse_error
0,1,phi-3.5-mini-instruct,Phi-3.5-mini-instruct,8000,v1_strict_json,high_risk,high_risk,"{""prediction"": ""high_risk"", ""confidence"": 0.95}",high_risk,0.95,True,None
1,2,phi-3.5-mini-instruct,Phi-3.5-mini-instruct,8000,v1_strict_json,low_risk,low_risk,"{""prediction"": ""low_risk"", ""confidence"": 0.95}",low_risk,0.95,True,None
2,3,phi-3.5-mini-instruct,Phi-3.5-mini-instruct,8000,v1_strict_json,moderate_risk,moderate_risk,"{""prediction"": ""moderate_risk"", ""confidence"":...",moderate_risk,0.75,True,None
3,4,phi-3.5-mini-instruct,Phi-3.5-mini-instruct,8000,v1_strict_json,high_risk,high_risk,"{""prediction"": ""high_risk"", ""confidence"": 0.95}",high_risk,0.95,True,None
4,5,phi-3.5-mini-instruct,Phi-3.5-mini-instruct,8000,v1_strict_json,low_risk,low_risk,"{""prediction"": ""low_risk"", ""confidence"": 0.95}",low_risk,0.95,True,None


## Step 7: Evaluate and compare

What this does: computes ML accuracy, LLM accuracy, and invalid output rate from the parsed results.

Why it matters: this gives a direct Week 3 vs Week 4 comparison on the same data split.

Principle: evaluate both quality and reliability. Accuracy without format reliability can still fail in production workflows.

In [11]:
eval_df = results_df.copy()
eval_df['llm_prediction_filled'] = eval_df['llm_prediction'].fillna('__INVALID__')

ml_accuracy = float((eval_df['ground_truth'] == eval_df['ml_prediction']).mean())

llm_summary = eval_df.groupby(['model_label', 'model_name', 'endpoint_port'], as_index=False).agg(
    llm_accuracy=('llm_prediction_filled', lambda s: float((eval_df.loc[s.index, 'ground_truth'] == s).mean())),
    invalid_output_rate=('parse_ok', lambda s: float((~s).mean())),
    rows=('model_name', 'size')
)

summary_df = pd.concat([
    pd.DataFrame([{'model_type': 'Week3_ML', 'model_label': 'week3_ml', 'model_name': 'Week3 ML Baseline', 'endpoint_port': np.nan, 'llm_accuracy': ml_accuracy, 'invalid_output_rate': 0.0, 'rows': len(eval_df)}]),
    llm_summary.assign(model_type='Week4_LLM')
] , ignore_index=True)

display(summary_df)
display(eval_df[['model_name', 'ground_truth', 'ml_prediction', 'llm_prediction', 'parse_ok', 'parse_error']].head(12))

,model_type,model_label,model_name,endpoint_port,llm_accuracy,invalid_output_rate,rows
0,Week3_ML,week3_ml,Week3 ML Baseline,NaN,0.975,0.0,120
1,Week4_LLM,gemma-3-12b-it,gemma-3-12b-it,8002.0,0.950,0.0,40
2,Week4_LLM,phi-3.5-mini-instruct,Phi-3.5-mini-instruct,8000.0,1.000,0.0,40
3,Week4_LLM,phi-mini-moe-instruct,Phi-mini-MoE-instruct,8001.0,0.950,0.0,40


,model_name,ground_truth,ml_prediction,llm_prediction,parse_ok,parse_error
0,Phi-3.5-mini-instruct,high_risk,high_risk,high_risk,True,None
1,Phi-3.5-mini-instruct,low_risk,low_risk,low_risk,True,None
2,Phi-3.5-mini-instruct,moderate_risk,moderate_risk,moderate_risk,True,None
3,Phi-3.5-mini-instruct,high_risk,high_risk,high_risk,True,None
4,Phi-3.5-mini-instruct,low_risk,low_risk,low_risk,True,None
5,Phi-3.5-mini-instruct,moderate_risk,low_risk,moderate_risk,True,None
6,Phi-3.5-mini-instruct,high_risk,high_risk,high_risk,True,None
7,Phi-3.5-mini-instruct,low_risk,low_risk,low_risk,True,None
8,Phi-3.5-mini-instruct,moderate_risk,moderate_risk,moderate_risk,True,None
9,Phi-3.5-mini-instruct,high_risk,high_risk,high_risk,True,None


## Step 8: Save clean CSV and draft summary

What this does: writes a clean submission-ready CSV and generates a short comparison paragraph for reporting.

Why it matters: deliverables should be generated by code to stay consistent and repeatable.

Principle: automate outputs for reproducibility. If you rerun the notebook, you should get the same structure every time.

In [12]:
clean_cols = [
    ID_COL, 'model_label', 'model_name', 'endpoint_port', 'prompt_version', 'ground_truth', 'ml_prediction',
    'llm_prediction', 'llm_confidence', 'parse_ok', 'parse_error'
 ]
clean_df = results_df[clean_cols].copy()
clean_df.to_csv(CLEAN_RESULTS_PATH, index=False)

# Also export per-model files following competition naming style.
for model_label, part in clean_df.groupby('model_label'):
    per_model_clean = OUTPUT_DIR / f'{model_label}_results_clean_{ITERATION_NUMBER}.csv'
    per_model_raw = OUTPUT_DIR / f'{model_label}_results_raw_{ITERATION_NUMBER}.csv'
    part.to_csv(per_model_clean, index=False)
    results_df[results_df['model_label'] == model_label].to_csv(per_model_raw, index=False)
    print(f'Saved per-model clean CSV: {per_model_clean}')
    print(f'Saved per-model raw CSV: {per_model_raw}')

print(f'\nSaved combined clean CSV: {CLEAN_RESULTS_PATH}')
print(f'Saved combined raw CSV:  {RAW_RESULTS_PATH}')

print('\nComparison Summary draft (per model):')
for _, row in llm_summary.iterrows():
    print(
        f"Week 3 ML accuracy={ml_accuracy:.3f}; "
        f"{row['model_name']} (port {int(row['endpoint_port'])}) accuracy={row['llm_accuracy']:.3f}; "
        f"invalid output rate={row['invalid_output_rate']:.3f}."
    )

Saved per-model clean CSV: /home/jupyter-abby/Results/gemma-3-12b-it_results_clean_1.csv
Saved per-model raw CSV: /home/jupyter-abby/Results/gemma-3-12b-it_results_raw_1.csv
Saved per-model clean CSV: /home/jupyter-abby/Results/phi-3.5-mini-instruct_results_clean_1.csv
Saved per-model raw CSV: /home/jupyter-abby/Results/phi-3.5-mini-instruct_results_raw_1.csv
Saved per-model clean CSV: /home/jupyter-abby/Results/phi-mini-moe-instruct_results_clean_1.csv
Saved per-model raw CSV: /home/jupyter-abby/Results/phi-mini-moe-instruct_results_raw_1.csv

Saved combined clean CSV: /home/jupyter-abby/Results/llm_results_clean_1.csv
Saved combined raw CSV:  /home/jupyter-abby/Results/llm_results_raw_1.csv

Comparison Summary draft (per model):
Week 3 ML accuracy=0.975; gemma-3-12b-it (port 8002) accuracy=0.950; invalid output rate=0.000.
Week 3 ML accuracy=0.975; Phi-3.5-mini-instruct (port 8000) accuracy=1.000; invalid output rate=0.000.
Week 3 ML accuracy=0.975; Phi-mini-MoE-instruct (port 8001) 

### TRY THIS

- Modify the prompt wording and check parse reliability
- Add few-shot examples to the prompt
- Add macro-F1 so it matches your Week 3 metrics exactly
